# U-Net Segmentation - Train & Apply

In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)

## Config

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
SEED = 42
EPOCHS = 30

DATASET_ROOT = Path('Dataset_Split_Morphological')
OUTPUT_ROOT = Path('Dataset_Split_UNet')
OUTPUTS_DIR = Path('outputs')
MODELS_DIR = OUTPUTS_DIR / 'models_unet'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(exist_ok=True)

np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Dataset: {DATASET_ROOT}")
print(f"Output: {OUTPUT_ROOT}")
print(f"Models: {MODELS_DIR}")

## Build U-Net Architecture

In [ ]:
def build_unet(input_size=(224, 224, 3)):
    inputs = layers.Input(input_size)
    
    # Encoder
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    c1 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c1)
    p1 = layers.MaxPooling2D((2, 2))(c1)
    
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p1)
    c2 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c2)
    p2 = layers.MaxPooling2D((2, 2))(c2)
    
    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(p2)
    c3 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c3)
    p3 = layers.MaxPooling2D((2, 2))(c3)
    
    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(p3)
    c4 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c4)
    p4 = layers.MaxPooling2D((2, 2))(c4)
    
    # Bottleneck
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(p4)
    c5 = layers.Conv2D(1024, (3, 3), activation='relu', padding='same')(c5)
    
    # Decoder
    u6 = layers.UpSampling2D((2, 2))(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(u6)
    c6 = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(c6)
    
    u7 = layers.UpSampling2D((2, 2))(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(u7)
    c7 = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(c7)
    
    u8 = layers.UpSampling2D((2, 2))(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(u8)
    c8 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(c8)
    
    u9 = layers.UpSampling2D((2, 2))(c8)
    u9 = layers.concatenate([u9, c1])
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(u9)
    c9 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(c9)
    
    outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(c9)
    
    return models.Model(inputs, outputs)

unet_model = build_unet()
unet_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print("U-Net model built")

## Generate Training Data (Images + Morphological Masks)

In [ ]:
def segment_leaf_morphological(image):
    img_hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    
    lower_green = np.array([25, 40, 40])
    upper_green = np.array([90, 255, 255])
    mask_green = cv2.inRange(img_hsv, lower_green, upper_green)
    
    kernel_erode = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11))
    
    mask_green = cv2.erode(mask_green, kernel_erode, iterations=1)
    mask_green = cv2.dilate(mask_green, kernel_dilate, iterations=1)
    mask_green = cv2.morphologyEx(mask_green, cv2.MORPH_CLOSE, kernel_close)
    
    contours, _ = cv2.findContours(mask_green, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    mask_final = np.zeros(mask_green.shape, dtype=np.uint8)
    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        cv2.drawContours(mask_final, [largest_contour], 0, 255, -1)
    
    return mask_final / 255.0

print("Morphological segmentation function ready")

## Load & Prepare Training Data

In [ ]:
train_images = []
train_masks = []

train_dir = DATASET_ROOT / 'train'
total_images = sum(1 for _ in train_dir.glob('*/*.jpg')) + sum(1 for _ in train_dir.glob('*/*.png')) + sum(1 for _ in train_dir.glob('*/*.jpeg'))

count = 0
for class_dir in sorted(train_dir.iterdir()):
    if class_dir.is_dir():
        images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpeg'))
        for img_path in tqdm(images, desc=f'Loading {class_dir.name}'):
            img = cv2.imread(str(img_path))
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img_resized = cv2.resize(img_rgb, IMG_SIZE)
            
            # Generate mask
            mask = segment_leaf_morphological(img_resized)
            
            train_images.append(img_resized.astype(np.float32) / 255.0)
            train_masks.append(mask)
            
            count += 1

train_images = np.array(train_images)
train_masks = np.array(train_masks)

print(f"\nLoaded {len(train_images)} training images")
print(f"Image shape: {train_images.shape}")
print(f"Mask shape: {train_masks.shape}")

## Visualize Training Data

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for i in range(9):
    ax = axes.flatten()[i]
    img = train_images[i]
    mask = train_masks[i]
    
    combined = np.hstack([img, np.stack([mask]*3, axis=-1)])
    ax.imshow(combined)
    ax.set_title(f'Image | Mask {i}')
    ax.axis('off')

plt.suptitle('Training Data: Original Images & Morphological Masks', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Train U-Net

In [ ]:
from tensorflow.keras import callbacks

unet_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint(
        str(MODELS_DIR / 'unet_best.keras'),
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=0
    ),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-7, verbose=1)
]

print("Training U-Net...\n")
unet_history = unet_model.fit(
    train_images,
    train_masks,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=unet_callbacks,
    verbose=1
)

print("\nU-Net training complete")

## Load Best U-Net Model

In [ ]:
unet_best = tf.keras.models.load_model(str(MODELS_DIR / 'unet_best.keras'))
print("Best U-Net model loaded")

## Apply U-Net to All Splits

In [ ]:
def apply_unet_segmentation(image_path, unet_model):
    img = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, IMG_SIZE)
    img_normalized = img_resized.astype(np.float32) / 255.0
    
    # Predict mask
    mask = unet_model.predict(np.expand_dims(img_normalized, axis=0), verbose=0)[0, :, :, 0]
    mask = (mask > 0.5).astype(np.uint8) * 255
    
    # Apply mask to image
    segmented = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)
    
    return segmented

for split in ['train', 'val', 'test']:
    split_dir = DATASET_ROOT / split
    output_split_dir = OUTPUT_ROOT / split
    
    for class_dir in split_dir.iterdir():
        if class_dir.is_dir():
            class_name = class_dir.name
            output_class_dir = output_split_dir / class_name
            output_class_dir.mkdir(parents=True, exist_ok=True)
            
            images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpeg'))
            
            for img_path in tqdm(images, desc=f'{split}/{class_name}'):
                segmented = apply_unet_segmentation(img_path, unet_best)
                
                output_path = output_class_dir / img_path.name
                segmented_bgr = cv2.cvtColor(segmented, cv2.COLOR_RGB2BGR)
                cv2.imwrite(str(output_path), segmented_bgr)

print("\nU-Net segmentation applied to all splits")

## Verify Output

In [ ]:
split_counts = {}

for split in ['train', 'val', 'test']:
    split_counts[split] = {}
    split_dir = OUTPUT_ROOT / split
    
    for class_dir in split_dir.iterdir():
        if class_dir.is_dir():
            images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png')) + list(class_dir.glob('*.jpeg'))
            split_counts[split][class_dir.name] = len(images)

total_unet = sum(sum(counts.values()) for counts in split_counts.values())

print("\n" + "="*60)
print("U-NET SEGMENTATION COMPLETE")
print("="*60)
for split in ['train', 'val', 'test']:
    count = sum(split_counts[split].values())
    print(f"{split}: {count} images")

print(f"\nTotal: {total_unet} segmented images")
print(f"Output directory: {OUTPUT_ROOT}")
print("="*60)

## Visualize Results

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
axes = axes.flatten()

idx = 0
for class_dir in (OUTPUT_ROOT / 'train').iterdir():
    if idx >= len(axes):
        break
    if class_dir.is_dir():
        images = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
        if images:
            img = cv2.imread(str(images[0]))
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[idx].imshow(img_rgb)
            axes[idx].set_title(class_dir.name, fontsize=9)
            axes[idx].axis('off')
            idx += 1

for j in range(idx, len(axes)):
    axes[j].axis('off')

plt.suptitle('U-Net Segmented Images (Sample)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(unet_history.history['accuracy'], label='Train Accuracy')
axes[0].plot(unet_history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('U-Net Training Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(unet_history.history['loss'], label='Train Loss')
axes[1].plot(unet_history.history['val_loss'], label='Val Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].set_title('U-Net Training Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'unet_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

print("Training history saved")